# Localized question difficulty — primary models

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
def find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        q=p/"tasks/political-compass/qualitative-analysis"
        if (q/"config.py").exists(): return q
        if p.name=="qualitative-analysis" and (p/"config.py").exists(): return p
    raise FileNotFoundError
ROOT=find_root(); TABLES=ROOT/"artifacts/tables"; FIGURES=ROOT/"artifacts/figures"
CHAT_ORDER=["gemma-3-1b-it","gemma-3-4b-it","gemma-3-12b-it","gemma-3-27b-it",
"Qwen3-4B_no_think","Qwen3-8B_no_think","Qwen3-14B_no_think","Qwen3-32B_no_think",
"Qwen3-4B_think","Qwen3-8B_think","Qwen3-14B_think","Qwen3-32B_think"]
LABEL=dict(zip(CHAT_ORDER,["Gemma 1B","Gemma 4B","Gemma 12B","Gemma 27B",
"Qwen 4B","Qwen 8B","Qwen 14B","Qwen 32B","Qwen 4B Think","Qwen 8B Think",
"Qwen 14B Think","Qwen 32B Think"]))
def ordered(d,col="model_variant"):
    d=d[d[col].isin(CHAT_ORDER)].copy(); d["model_label"]=d[col].map(LABEL)
    d["model_label"]=pd.Categorical(d.model_label,[LABEL[x] for x in CHAT_ORDER],ordered=True)
    return d.sort_values("model_label")
sns.set_theme(style="whitegrid")


DIF identifies a model/persona/question that is unusually easy or difficult after accounting for general model/persona performance and general question difficulty. It does not establish one universally hardest quadrant.

In [2]:
d=ordered(pd.read_csv(TABLES/"question_dif_family_protocol.csv")); d=d[d.ideology.isin(["libertarian_left","libertarian_right","authoritarian_left","authoritarian_right"])]
d["localized"]=d.dif_abs>=1; d["direction"]=np.where(d.dif_logit<0,"unusually difficult","unusually easy")
display(d.sort_values("dif_abs",ascending=False)[["model_label","ideology","question_id","topic","target_alignment_rate","dif_logit","direction"]].head(60))
r=d[d.localized].groupby(["ideology","question_id","topic","direction"]).agg(variants=("model_variant","nunique"),gemma=("family",lambda x:int((x=="Gemma-3").sum())),qwen=("family",lambda x:int((x=="Qwen3").sum())),median_dif=("dif_logit","median")).reset_index()
display(r.sort_values(["variants","median_dif"],ascending=[False,True]).head(40))

,model_label,ideology,question_id,topic,target_alignment_rate,dif_logit,direction
1,Qwen 8B Think,libertarian_right,34,welfare_work,0.000000,-7.058411,unusually difficult
0,Qwen 32B Think,libertarian_right,34,welfare_work,0.000000,-7.058411,unusually difficult
2,Qwen 4B Think,libertarian_right,7,class_nationality,0.003333,-6.483888,unusually difficult
3,Qwen 14B Think,authoritarian_left,51,pseudoscience,0.000000,-6.302465,unusually difficult
4,Qwen 8B Think,libertarian_right,45,punishment_justice,0.003333,-6.297762,unusually difficult
6,Qwen 14B Think,libertarian_right,34,welfare_work,0.003333,-5.956466,unusually difficult
5,Qwen 4B Think,libertarian_right,34,welfare_work,0.003333,-5.956466,unusually difficult
7,Qwen 8B Think,authoritarian_left,45,punishment_justice,1.000000,5.879386,unusually easy
8,Qwen 14B Think,authoritarian_left,52,religion_morality,0.003333,-5.858800,unusually difficult
9,Qwen 8B Think,authoritarian_left,19,consumer_regulation,1.000000,5.853830,unusually easy


,ideology,question_id,topic,direction,variants,gemma,qwen,median_dif
317,libertarian_right,34,welfare_work,unusually difficult,11,3,8,-3.706838
332,libertarian_right,45,punishment_justice,unusually difficult,11,3,8,-2.242167
199,libertarian_left,15,trade_protectionism,unusually difficult,11,3,8,-2.137905
82,authoritarian_left,57,lgbt_family,unusually difficult,11,3,8,-1.885921
10,authoritarian_left,7,class_nationality,unusually easy,11,3,8,1.921726
338,libertarian_right,51,pseudoscience,unusually easy,11,3,8,2.531295
122,authoritarian_right,27,family_discipline,unusually easy,11,3,8,2.672879
304,libertarian_right,24,culture_public_funding,unusually easy,11,3,8,2.716476
350,libertarian_right,59,sexuality_privacy,unusually easy,11,3,8,2.742048
348,libertarian_right,58,sexuality_expression,unusually easy,11,3,8,2.745988
